In [10]:
%%writefile requirements.txt

pydantic

Overwriting requirements.txt


In [7]:
!pip install -r requirements.txt


[notice] A new release of pip is available: 23.0.1 -> 26.3
[notice] To update, run: pip install --upgrade pip


In [12]:
%pip install pydantic

  Using cached pydantic-2.14.3-py3-none-any.whl.metadata (141 kB)
  Using cached annotated_types-0.8.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_inspection-0.4.5-py3-none-any.whl.metadata (2.6 kB)
Using cached pydantic-2.14.3-py3-none-any.whl (494 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 41.4 kB/s  0:00:416.7 kB/s eta 0:00:03:06
Using cached annotated_types-0.8.0-py3-none-any.whl (13 kB)
Using cached typing_inspection-0.4.5-py3-none-any.whl (14 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [pydantic]━━ 3/4 [pydantic]
Note: you may need to restart the kernel to use updated packages.


In [2]:
# import python libraries first
from pydantic import BaseModel
from datetime import date

class Automobile(BaseModel):
    manufacturer: str
    series_name: str
    type_: str
    is_electric: bool = False
    manufactured_date: date
    base_msrp_usd: float
    vin: str
    number_of_doors: int = 4
    registration_country: str | None = None
    license_plate: str | None = None

In [10]:
# Python dictionary

data = {
    "manufacturer": "BMW",
    "series_name": "M4",
    "type_": "Convertible",
    "is_electric": False,
    "manufactured_date": "2023-01-01",
    "base_msrp_usd": 93_300,
    "vin": "1234567890",
    "number_of_doors": 2,
    "registration_country": "France",
    "license_plate": "AAA-BBB",
}

data_expected_serialization = {
    'manufacturer': 'BMW',
    'series_name': 'M4',
    'type_': 'Convertible',
    'is_electric': False,
    'manufactured_date': date(2023,1,1),
    'base_msrp_usd': 93_300,
    'vin': '1234567890',
    'number_of_doors': 2,
    'registration_country': 'France',
    'license_plate': 'AAA-BBB',
}

In [11]:
# JSON
data_json = '''
{
    "manufacturer": "BMW",
    "series_name": "M4",
    "type_": "Convertible",
    "manufactured_date": "2023-01-01",
    "base_msrp_usd": 93300,
    "vin": "1234567890"
}
'''

data_json_expected_serialization = {
    'manufacturer': 'BMW',
    'series_name': 'M4',
    'type_': 'Convertible',
    'is_electric': False,
    'manufactured_date': date(2023, 1, 1),
    'base_msrp_usd': 93_300,
    'vin': '1234567890',
    'number_of_doors': 4,
    'registration_country': None,
    'license_plate': None,
}

In [12]:
auto1 = Automobile.model_validate(data)

In [14]:
assert auto1.model_dump() == data_expected_serialization

In [15]:
auto2 = Automobile.model_validate_json(data_json)

In [16]:
assert auto2.model_dump() == data_json_expected_serialization

In [17]:
from pydantic import ValidationError
data_invalid = data.copy()
data_invalid['is_electric'] = 100

try:
    Automobile.model_validate(data_invalid)
except ValidationError as e:
    print(e)
    assert "Input should be a valid boolean" in str(e)

data_invalid = data.copy()
del data_invalid['manufacturer']
try:
    Automobile.model_validate(data_invalid)
except ValidationError as e:
    print(e)
    assert "Field required" in str(e)

1 validation error for Automobile
is_electric
  Input should be a valid boolean, unable to interpret input [type=bool_parsing, input_value=100, input_type=int]
    For further information visit https://errors.pydantic.dev/2.14/v/bool_parsing
1 validation error for Automobile
manufacturer
  Field required [type=missing, input_value={'series_name': 'M4', 'ty...cense_plate': 'AAA-BBB'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing


In [20]:
from enum import Enum

class AutomobileType(Enum):
    sedan = "Sedan"
    coupe = "Coupe"
    convertible = "Convertible"
    suv = "SUV"
    truck = "Truck"

In [22]:
data_json = '''
{
    "manufacturer": " BMW ",
    "series_name": " M4 ",
    "type_": "Convertible",
    "manufactured_date": "2023-01-01",
    "base_msrp_usd": 93300,
    "vin": " 1234567890 "
}
'''

data_json_expected_serialization = {
    'manufacturer': 'BMW',
    'series_name': 'M4',
    'type_': AutomobileType.convertible,
    'is_electric': False,
    'manufactured_date': date(2023, 1, 1),
    'base_msrp_usd': 93300.0,
    'vin': '1234567890',
    'number_of_doors': 4,
    'registration_country': None,
    'license_plate': None
}

In [25]:
from pydantic import ConfigDict

class Automobile(BaseModel):
    model_config = ConfigDict(extra='forbid',
                              str_strip_whitespace=True,
                              validate_default=True,
                              validate_assignment=True)
    manufacturer: str
    series_name: str
    type_: AutomobileType
    is_electric : bool = False
    manufactured_date: date
    base_msrp_usd : float
    vin : str
    number_of_doors : int = 4
    registration_country : str | None = None
    license_plate : str | None = None

In [27]:
m = Automobile.model_validate_json(data_json)
assert m.model_dump() == data_json_expected_serialization

In [29]:
from pydantic import ValidationError
data_invalid = data.copy()
data_invalid['extra_field'] = 'extra'
try:
    Automobile.model_validate(data_invalid)
except ValidationError as e:
    print(e)
    assert "Extra inputs are not permitted" in str(e)

1 validation error for Automobile
extra_field
  Extra inputs are not permitted [type=extra_forbidden, input_value='extra', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/extra_forbidden


In [31]:


data_json = '''
{
    "manufacturer": "BMW",
    "seriesName": "M4",
    "type": "Convertible",
    "isElectric": false,
    "completionDate": "2023-01-01",
    "msrpUSD": 93300,
    "vin": "1234567890",
    "doors": 2,
    "registrationCountry": "France",
    "licensePlate": "AAA-BBB"
}
'''



In [33]:


expected_serialized_dict = {
    'manufacturer': 'BMW',
    'series_name': 'M4',
    'type_': AutomobileType.convertible,
    'is_electric': False,
    'manufactured_date': date(2023, 1, 1),
    'base_msrp_usd': 93300.0,
    'vin': '1234567890',
    'number_of_doors': 2,
    'registration_country': 'France',
    'license_plate': 'AAA-BBB'
}

expected_serialized_dict_by_alias = {
    'manufacturer': 'BMW',
    'seriesName': 'M4',
    'type': AutomobileType.convertible,
    'isElectric': False,
    'manufacturedDate': date(2023, 1, 1),
    'baseMSRPUSD': 93300.0,
    'vin': '1234567890',
    'numberOfDoors': 2,
    'registrationCountry': 'France',
    'licensePlate': 'AAA-BBB'
}

expected_serialized_json_by_alias = (
    '{"manufacturer":"BMW","seriesName":"M4","type":"Convertible",'
    '"isElectric":false,"manufacturedDate":"2023/01/01","baseMSRPUSD":93300.0,'
    '"vin":"1234567890","numberOfDoors":2,"registrationCountry":"France",'
    '"licensePlate":"AAA-BBB"}'
)



In [35]:
from pydantic.alias_generators import to_camel
from pydantic import Field, field_serializer
class Automobile(BaseModel):
    model_config = ConfigDict(extra='forbid',
                              str_strip_whitespace=True,
                              validate_default=True,
                              validate_assignment=True, alias_generator=to_camel)
    manufacturer: str
    series_name: str
    type_: AutomobileType = Field(alias='type')
    is_electric : bool = False
    manufactured_date: date = Field(validation_alias='completionDate')
    base_msrp_usd : float = Field(validation_alias='msrpUSD', serialization_alias='baseMSRPUSD')
    vin : str
    number_of_doors : int = Field(default=4, validation_alias="doors")
    registration_country : str | None = None
    license_plate : str | None = None
    
    @field_serializer("manufactured_date", when_used="json-unless-none")
    def serialize_manufactured_date(self, value:date):
        return value.strftime("%Y/%m/%d")


In [37]:
m = Automobile.model_validate_json(data_json)

In [39]:
assert m.model_dump() == expected_serialized_dict

In [41]:
assert m.model_dump(by_alias=True) == expected_serialized_dict_by_alias

In [43]:
assert m.model_dump_json(by_alias=True) == expected_serialized_json_by_alias

In [45]:
from uuid import UUID

data = {
    "id": "c4e60f4a-3c7f-4da5-9b3f-07aee50b23e7",
    "manufacturer": "BMW",
    "seriesName": "M4",
    "type": "Convertible",
    "isElectric": False,
    "completionDate": "2023-01-01",
    "msrpUSD": 93_300,
    "vin": "1234567890",
    "doors": 2,
    "registrationCountry": "France",
    "licensePlate": "AAA-BBB"
}

expected_serialized_by_alias = {
    'id': UUID('c4e60f4a-3c7f-4da5-9b3f-07aee50b23e7'),
    'manufacturer': 'BMW',
    'seriesName': 'M4',
    'type': AutomobileType.convertible,
    'isElectric': False,
    'manufacturedDate': date(2023, 1, 1),
    'baseMSRPUSD': 93300.0,
    'vin': '1234567890',
    'numberOfDoors': 2,
    'registrationCountry': 'France',
    'licensePlate': 'AAA-BBB'
}

data_no_id = {
    "manufacturer": "BMW",
    "seriesName": "M4",
    "type": "Convertible",
    "isElectric": False,
    "completionDate": "2023-01-01",
    "msrpUSD": 93_300,
    "vin": "1234567890",
    "doors": 2,
    "registrationCountry": "France",
    "licensePlate": "AAA-BBB"
}

expected_serialization_data_no_id_by_alias = {
    'id': None,
    'manufacturer': 'BMW',
    'seriesName': 'M4',
    'type': AutomobileType.convertible,
    'isElectric': False,
    'manufacturedDate': date(2023, 1, 1),
    'baseMSRPUSD': 93300.0,
    'vin': '1234567890',
    'numberOfDoors': 2,
    'registrationCountry': 'France',
    'licensePlate': 'AAA-BBB'
}

In [46]:
from pydantic import UUID4
class Automobile(BaseModel):
    model_config = ConfigDict(extra='forbid',
                              str_strip_whitespace=True,
                              validate_default=True,
                              validate_assignment=True, alias_generator=to_camel)
    id_: UUID4 | None = Field(alias='id',default=None)
    manufacturer: str
    series_name: str
    type_: AutomobileType = Field(alias='type')
    is_electric : bool = False
    manufactured_date: date = Field(validation_alias='completionDate')
    base_msrp_usd : float = Field(validation_alias='msrpUSD', serialization_alias='baseMSRPUSD')
    vin : str
    number_of_doors : int = Field(default=4, validation_alias="doors")
    registration_country : str | None = None
    license_plate : str | None = None
    
    @field_serializer("manufactured_date", when_used="json-unless-none")
    def serialize_manufactured_date(self, value:date):
        return value.strftime("%Y/%m/%d")